In [1]:
"""
================================================================================
Automated Batch Export of JRC Yearly Water History for Pristine Mangrove Tiles
- Uses the GEE Python API to bypass browser UI limits for large datasets (1300+ tiles).
- Exports multi-band GeoTIFFs (1985-2025) directly to Google Drive.
================================================================================
"""

import ee
import geopandas as gpd

# 1. Initialize Earth Engine (make sure you have run 'earthengine authenticate' beforehand)
try:
    ee.Initialize()
except Exception as e:
    print("Initializing and authenticating Earth Engine...")
    ee.Authenticate()
    ee.Initialize()

# 2. Load the vector asset of pristine tiles from GEE
ASSET_ID = "projects/amibea/assets/gmwv4_mask_anthropogenic_validation"
print(f"📂 Loading GEE asset: {ASSET_ID}")
validation_asset = ee.FeatureCollection(ASSET_ID)

# Filter strictly to keep only pristine tiles (mask_artif == 0)
pristine_tiles = validation_asset.filter(ee.Filter.eq('mask_artif', 0))

# Load the JRC Yearly Water History collection
gsw_yearly = ee.ImageCollection('JRC/GSW1_4/YearlyHistory')

# 3. Retrieve the list of tiles client-side via getInfo()
collection_geojson = pristine_tiles.getInfo()
features_list = collection_geojson['features']

total_tiles = len(features_list)
print(f"🚀 Total pristine tiles to submit in batch: {total_tiles}")

# 4. Loop through each tile and submit the export task to Google Drive
tasks_submitted = 0

for idx, feat in enumerate(features_list):
    props = feat['properties']
    tile_id = props.get('tile_id', f'tile_{idx}')
    
    # Extract tile geometry
    geom_json = feat['geometry']
    geom = ee.Geometry(geom_json)
    
    # Convert the image collection directly into a multi-band image stack
    stack = gsw_yearly.toBands()
    
    # Define a unique task description
    task_description = f"JRC_Yearly_{tile_id}"
    
    # Configure the export task to Google Drive
    task = ee.batch.Export.image.toDrive(
        image=stack,
        description=task_description,
        folder='JRC_Mangroves_Pristine_Tiles',
        fileNamePrefix=f"JRC_Yearly_{tile_id}",
        region=geom,
        scale=100,          # Adjust to 30 for native resolution if needed
        maxPixels=1e13,
        crs='EPSG:4326'
    )
    
    # Start the task
    task.start()
    tasks_submitted += 1
    
    print(f"[{tasks_submitted}/{total_tiles}] Task submitted for tile: {tile_id}")

print("\n✅ All export tasks have been successfully submitted to Google Earth Engine!")
print("💡 You can monitor their progress in the Tasks tab of the GEE Code Editor web interface.")

📂 Loading GEE asset: projects/amibea/assets/gmwv4_mask_anthropogenic_validation
🚀 Total pristine tiles to submit in batch: 1261
[1/1261] Task submitted for tile: S15E136
[2/1261] Task submitted for tile: S20E117
[3/1261] Task submitted for tile: S38E146
[4/1261] Task submitted for tile: S13E130
[5/1261] Task submitted for tile: S18E122
[6/1261] Task submitted for tile: S19E119
[7/1261] Task submitted for tile: S36E174
[8/1261] Task submitted for tile: S33E137
[9/1261] Task submitted for tile: S11E134
[10/1261] Task submitted for tile: S16E124
[11/1261] Task submitted for tile: S13E129
[12/1261] Task submitted for tile: S29E153
[13/1261] Task submitted for tile: S33E151
[14/1261] Task submitted for tile: S14E143
[15/1261] Task submitted for tile: S16E138
[16/1261] Task submitted for tile: S13E135
[17/1261] Task submitted for tile: S14E125
[18/1261] Task submitted for tile: S38E177
[19/1261] Task submitted for tile: S15E137
[20/1261] Task submitted for tile: S11E130
[21/1261] Task submit

In [1]:
"""
================================================================================
Batch reprojects multi-band JRC Global Surface Water (GSW) yearly history 
rasters from their original CRS to the ESRI:54034 (Cylindrical Equal Area) 
coordinate reference system using rasterio. Implements smart skipping for 
already processed files and uses nearest-neighbor resampling to preserve 
discrete water classification classes.
================================================================================
"""

from pathlib import Path
import rasterio
from rasterio.warp import calculate_default_transform, reproject, Resampling
from rasterio.crs import CRS

# ==========================================================================
# 1. PATHS & CONFIGURATION
# ==========================================
JRC_RAW_DIR = Path(r"D:\MANGROVES\JRC_GSW\JRC_GSW1_4_YearlyHistory")
JRC_REPROJ_DIR = JRC_RAW_DIR / "reprojected_54034"
JRC_REPROJ_DIR.mkdir(parents=True, exist_ok=True)

# Define target spatial reference system (Cylindrical Equal Area) and NoData value
TARGET_CRS = CRS.from_string("ESRI:54034")
NODATA_VAL = 0

# ==========================================================================
# 2. IDENTIFY PENDING RASTERS TO REPROJECT
# ==========================================
all_raw_files = list(JRC_RAW_DIR.glob("JRC_Yearly_*.tif"))
print(f"🚀 Found {len(all_raw_files)} total raw JRC raster(s) in source folder.")

# Filter out files that have already been reprojected to avoid redundant processing
pending_files = []
for src_path in all_raw_files:
    out_path = JRC_REPROJ_DIR / f"{src_path.stem}_54034.tif"
    if not out_path.exists():
        pending_files.append(src_path)

print(f"🎯 {len(pending_files)} raster(s) remaining to reproject.")

# ==========================================================================
# 3. REPROJECT ALL BANDS (MULTI-BAND SUPPORT)
# ==========================================
for i, src_path in enumerate(pending_files, 1):
    out_path = JRC_REPROJ_DIR / f"{src_path.stem}_54034.tif"

    print(f"[{i}/{len(pending_files)}] ⚙️ Reprojecting multi-band raster {src_path.name} to ESRI:54034 ...")

    try:
        with rasterio.open(src_path) as src:
            # Calculate output dimensions, bounds, and transform matrix for the target CRS
            transform, width, height = calculate_default_transform(
                src.crs,
                TARGET_CRS,
                src.width,
                src.height,
                *src.bounds
            )

            # Copy and update metadata for the reprojected raster
            kwargs = src.meta.copy()
            kwargs.update({
                "crs": TARGET_CRS,
                "transform": transform,
                "width": width,
                "height": height,
                "nodata": NODATA_VAL,
                "dtype": "uint8",
                "compress": "lzw"
            })

            # Open output file preserving the exact original band count (e.g., yearly history layers)
            with rasterio.open(out_path, "w", **kwargs) as dst:
                # Iterate through each individual band (representing yearly history from 1985 to 2025)
                for band_idx in range(1, src.count + 1):
                    reproject(
                        source=rasterio.band(src, band_idx),
                        destination=rasterio.band(dst, band_idx),
                        src_transform=src.transform,
                        src_crs=src.crs,
                        dst_transform=transform,
                        dst_crs=TARGET_CRS,
                        # Nearest neighbor resampling is critical to maintain discrete water classes (0, 1, 2, 3)
                        resampling=Resampling.nearest,
                        src_nodata=NODATA_VAL,
                        dst_nodata=NODATA_VAL
                    )

        print(f"  ✅ Successfully saved multi-band raster: {out_path.name}")

    except Exception as e:
        print(f"  ❌ Error reprojecting {src_path.name}: {e}")

print(f"\n🎉 Reprojection batch complete! All files are processed in {JRC_REPROJ_DIR}")

🚀 Found 1261 total raw JRC raster(s) in source folder.
🎯 999 raster(s) remaining to reproject.
[1/999] ⚙️ Reprojecting multi-band raster JRC_Yearly_N00E008.tif to ESRI:54034 ...
  ✅ Successfully saved multi-band raster: JRC_Yearly_N00E008_54034.tif
[2/999] ⚙️ Reprojecting multi-band raster JRC_Yearly_N00E009.tif to ESRI:54034 ...
  ✅ Successfully saved multi-band raster: JRC_Yearly_N00E009_54034.tif
[3/999] ⚙️ Reprojecting multi-band raster JRC_Yearly_N00E041.tif to ESRI:54034 ...
  ✅ Successfully saved multi-band raster: JRC_Yearly_N00E041_54034.tif
[4/999] ⚙️ Reprojecting multi-band raster JRC_Yearly_N00E042.tif to ESRI:54034 ...
  ✅ Successfully saved multi-band raster: JRC_Yearly_N00E042_54034.tif
[5/999] ⚙️ Reprojecting multi-band raster JRC_Yearly_N00E073.tif to ESRI:54034 ...
  ✅ Successfully saved multi-band raster: JRC_Yearly_N00E073_54034.tif
[6/999] ⚙️ Reprojecting multi-band raster JRC_Yearly_N00E097.tif to ESRI:54034 ...
  ✅ Successfully saved multi-band raster: JRC_Yearly